In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, round, row_number
from pyspark.sql.window import Window
import pandas as pd

# Inisialisasi SparkSession
spark = SparkSession.builder.appName("Tugas5").getOrCreate()

# 1. Baca transaksi_tugas5.csv dari HDFS dan tambahkan kolom 'pendapatan'
# Sesuaikan path HDFS dengan environment Anda (misal: "hdfs:///user/mahasiswa/tugas5/...")
df_transaksi = spark.read.csv("file:///home/azizi/praktikum-bigdata/transaksi_tugas5.csv", header=True, inferSchema=True)
df_transaksi = df_transaksi.withColumn("pendapatan", col("unit_terjual") * col("harga_satuan"))

# 2. Buat df_target dari dictionary
data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"],
}
# Konversi ke pandas dataframe lalu ke spark dataframe
df_target = spark.createDataFrame(pd.DataFrame(data_target_cabang))

In [6]:
# Kelompokkan total pendapatan per kota
df_pendapatan_kota = df_transaksi.groupBy("kota").agg(sum("pendapatan").alias("total_pendapatan"))

# Join dengan df_target
df_join_target = df_pendapatan_kota.join(df_target, on="kota", how="inner")

# Tambahkan kolom pencapaian_persen dan urutkan dari yang tertinggi
df_hasil_A = df_join_target.withColumn("pencapaian_persen", round((col("total_pendapatan") / col("target_bulanan")) * 100, 2)) \
                           .orderBy(col("pencapaian_persen").desc())

df_hasil_A.show()

[Stage 5:>                                                          (0 + 8) / 8]

+----------+----------------+--------------+----------+-----------------+
|      kota|total_pendapatan|target_bulanan|pic_cabang|pencapaian_persen|
+----------+----------------+--------------+----------+-----------------+
| Purworejo|        45650000|      30000000|     Fitri|           152.17|
|      Solo|        33475000|      40000000|      Bayu|            83.69|
|Yogyakarta|        47275000|      60000000|      Joko|            78.79|
|  Magelang|        31650000|      45000000|      Rani|            70.33|
|  Semarang|        38175000|      55000000|      Sari|            69.41|
+----------+----------------+--------------+----------+-----------------+



In [7]:
# Hitung total pendapatan per kota dan per kategori
df_kategori_kota = df_transaksi.groupBy("kota", "kategori").agg(sum("pendapatan").alias("total_pendapatan_kategori"))

# Definisikan Window untuk ranking
windowSpec = Window.partitionBy("kota").orderBy(col("total_pendapatan_kategori").desc())

# Tambahkan row_number dan filter top 1
df_ranked = df_kategori_kota.withColumn("rank", row_number().over(windowSpec))
df_hasil_B = df_ranked.filter(col("rank") == 1).drop("rank")

df_hasil_B.show()

+----------+--------------------+-------------------------+
|      kota|            kategori|total_pendapatan_kategori|
+----------+--------------------+-------------------------+
|  Magelang|Kesehatan & Kecan...|                  7275000|
| Purworejo|Kesehatan & Kecan...|                 10075000|
|  Semarang|        Rumah Tangga|                 11125000|
|      Solo|Kesehatan & Kecan...|                  8425000|
|Yogyakarta|             Fashion|                 13325000|
+----------+--------------------+-------------------------+



In [8]:
# Daftarkan sebagai temporary view
df_transaksi.createOrReplaceTempView("view_transaksi")
df_target.createOrReplaceTempView("view_target")

# Tulis satu kueri SQL
kueri_sql = """
SELECT 
    t.kota, 
    tg.pic_cabang, 
    COUNT(t.order_id) as jumlah_transaksi
FROM view_transaksi t
JOIN view_target tg ON t.kota = tg.kota
GROUP BY t.kota, tg.pic_cabang
ORDER BY jumlah_transaksi DESC
"""

df_hasil_C = spark.sql(kueri_sql)
df_hasil_C.show()

+----------+----------+----------------+
|      kota|pic_cabang|jumlah_transaksi|
+----------+----------+----------------+
| Purworejo|     Fitri|             116|
|Yogyakarta|      Joko|             110|
|      Solo|      Bayu|              95|
|  Semarang|      Sari|              93|
|  Magelang|      Rani|              86|
+----------+----------+----------------+



Dari hasil pengolahan data transaksi yang telah dilakukan, cabang Purworejo di bawah arahan Fitri berhasil menunjukkan kinerja yang sangat memuaskan dan memimpin di posisi pertama. Cabang ini sukses membukukan total pendapatan hingga Rp 45.650.000, yang berarti mereka berhasil menembus pencapaian target tertinggi di angka 152,17%. Keberhasilan ini sangat didorong oleh tingginya minat pelanggan pada kategori Kesehatan & Kecantikan, yang menjadi tulang punggung penjualan di kota tersebut dengan sumbangsih pendapatan mencapai Rp 10.075.000.
Di sisi lain, manajemen perlu memberikan pendampingan dan perhatian ekstra pada cabang Semarang. Kinerja cabang ini masih tertinggal dibandingkan yang lain, dengan total pendapatan yang baru terkumpul sebesar Rp 38.175.000. Angka tersebut menjadikan pencapaian target mereka berada di posisi terendah, yakni hanya 69,41%. Walaupun produk di kategori Rumah Tangga sebenarnya cukup diminati di wilayah ini, volume transaksi yang hanya menyentuh 93 pesanan belum cukup kuat untuk mengejar target keseluruhan. Ke depannya, manajemen sangat disarankan untuk meninjau ulang strategi pemasaran yang ada atau memberikan suntikan promosi khusus—terutama pada kategori produk lainnya—guna mendongkrak gairah penjualan di cabang ini. 